Для корректной работы с файлами из данного notebook используйте команду `%cd ..`

In [5]:
%cd ..

c:\Programming\PROJECTS\hackaton\trade-defense-platform


c:\Programming\PROJECTS\hackaton\trade-defense-platform\.venv\lib\site-packages\IPython\core\magics\osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [3]:
%pip install sentence_transformers pymupdf

## USER-BGE-3

In [6]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("deepvk/USER-base")

queries = ["Как приготовить борщ?", "Столица России"]
passages = [
    "Борщ - традиционное славянское блюдо. Для приготовления нужны свекла, капуста, картофель, морковь и мясной бульон.",
    "Пицца готовится из теста с томатным соусом и сыром.",
    "Москва является столицей Российской Федерации и крупнейшим городом страны.",
    "Париж - столица Франции, известная Эйфелевой башней."
]

query_embeddings = model.encode(queries, normalize_embeddings=True, prompt_name='query')
passage_embeddings = model.encode(passages, normalize_embeddings=True, prompt_name='passage')

scores = np.dot(query_embeddings, passage_embeddings.T) * 100

for i, query in enumerate(queries):
    print(f"\n{query}")
    for rank, idx in enumerate(np.argsort(scores[i])[::-1], 1):
        print(f"{rank}. [{scores[i][idx]:.1f}] {passages[idx]}")


Default prompt name is set to 'query'. This prompt will be applied to all `encode()` calls, except if `encode()` is called with `prompt` or `prompt_name` parameters.



Как приготовить борщ?
1. [46.2] Борщ - традиционное славянское блюдо. Для приготовления нужны свекла, капуста, картофель, морковь и мясной бульон.
2. [18.0] Пицца готовится из теста с томатным соусом и сыром.
3. [7.2] Москва является столицей Российской Федерации и крупнейшим городом страны.
4. [6.5] Париж - столица Франции, известная Эйфелевой башней.

Столица России
1. [59.6] Москва является столицей Российской Федерации и крупнейшим городом страны.
2. [34.6] Париж - столица Франции, известная Эйфелевой башней.
3. [20.0] Борщ - традиционное славянское блюдо. Для приготовления нужны свекла, капуста, картофель, морковь и мясной бульон.
4. [14.4] Пицца готовится из теста с томатным соусом и сыром.


# Qdrant

In [7]:
import pymupdf as fitz
import os
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
import uuid

pdf_folder = "data/rag"
model = SentenceTransformer("deepvk/USER-base")
client = QdrantClient(url="http://localhost:6333")
collection_name = "rag_docs"

def extract_text_from_pdfs(folder_path):
    documents = []
    for pdf_file in sorted(os.listdir(folder_path)):
        if not pdf_file.endswith('.pdf'):
            continue
        pdf_path = os.path.join(folder_path, pdf_file)
        doc = fitz.open(pdf_path)
        text = ""
        for page in doc:
            text += page.get_text()
        doc.close()
        documents.append({"filename": pdf_file, "text": text})
    return documents

def chunk_text(text, chunk_size=1500, overlap=200):
    chunks = []
    start = 0
    text_len = len(text)
    while start < text_len:
        end = start + chunk_size
        chunk = text[start:end]
        if chunk.strip():
            chunks.append(chunk)
        start = end - overlap
    return chunks

def create_collection(client, collection_name, vector_size):
    try:
        client.delete_collection(collection_name)
    except:
        pass
    client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(size=vector_size, distance=Distance.COSINE)
    )

documents = extract_text_from_pdfs(pdf_folder)
print(f"Загружено {len(documents)} документов")

all_chunks = []
for doc in documents:
    chunks = chunk_text(doc["text"])
    for chunk in chunks:
        all_chunks.append({"filename": doc["filename"], "text": chunk})

print(f"Создано {len(all_chunks)} чанков")

sample_embedding = model.encode(["тест"], normalize_embeddings=True, prompt_name='passage')[0]
vector_size = len(sample_embedding)

create_collection(client, collection_name, vector_size)

batch_size = 50
for i in range(0, len(all_chunks), batch_size):
    batch = all_chunks[i:i+batch_size]
    texts = [chunk["text"] for chunk in batch]
    embeddings = model.encode(texts, normalize_embeddings=True, prompt_name='passage')
    
    points = []
    for j, (chunk, embedding) in enumerate(zip(batch, embeddings)):
        points.append(PointStruct(
            id=str(uuid.uuid4()),
            vector=embedding.tolist(),
            payload={"filename": chunk["filename"], "text": chunk["text"]}
        ))
    
    client.upsert(collection_name=collection_name, points=points)
    print(f"Загружено {i+len(batch)}/{len(all_chunks)} чанков")

print(f"\nRAG хранилище готово!")
print(f"Коллекция: {collection_name}")
print(f"Документов: {len(documents)}")
print(f"Чанков: {len(all_chunks)}")

def search_rag(query, top_k=3):
    query_embedding = model.encode([query], normalize_embeddings=True, prompt_name='query')[0]
    results = client.search(
        collection_name=collection_name,
        query_vector=query_embedding.tolist(),
        limit=top_k
    )
    return results

print("\n" + "="*60)
print("ПРИМЕРЫ ПОИСКА:")
print("="*60)

test_queries = [
    "Таможенный кодекс ЕАЭС",
    "Распоряжение правительства 2022",
    "Механизм внесения изменений"
]

for query in test_queries:
    print(f"\nЗапрос: {query}")
    results = search_rag(query, top_k=3)
    for i, result in enumerate(results, 1):
        print(f"\n  {i}. Релевантность: {result.score:.3f}")
        print(f"     Файл: {result.payload['filename']}")
        print(f"     Текст: {result.payload['text'][:200]}...")


c:\Programming\PROJECTS\hackaton\trade-defense-platform\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Default prompt name is set to 'query'. This prompt will be applied to all `encode()` calls, except if `encode()` is called with `prompt` or `prompt_name` parameters.


Загружено 8 документов
Создано 1287 чанков
Загружено 50/1287 чанков
Загружено 100/1287 чанков
Загружено 150/1287 чанков
Загружено 200/1287 чанков
Загружено 250/1287 чанков
Загружено 300/1287 чанков
Загружено 350/1287 чанков
Загружено 400/1287 чанков
Загружено 450/1287 чанков
Загружено 500/1287 чанков
Загружено 550/1287 чанков
Загружено 600/1287 чанков
Загружено 650/1287 чанков
Загружено 700/1287 чанков
Загружено 750/1287 чанков
Загружено 800/1287 чанков
Загружено 850/1287 чанков
Загружено 900/1287 чанков
Загружено 950/1287 чанков
Загружено 1000/1287 чанков
Загружено 1050/1287 чанков
Загружено 1100/1287 чанков
Загружено 1150/1287 чанков
Загружено 1200/1287 чанков
Загружено 1250/1287 чанков
Загружено 1287/1287 чанков

RAG хранилище готово!
Коллекция: rag_docs
Документов: 8
Чанков: 1287

ПРИМЕРЫ ПОИСКА:

Запрос: Таможенный кодекс ЕАЭС

  1. Релевантность: 0.621
     Файл: 9 Часто задаваемые вопросы.pdf
     Текст: пошлины,
установленные на территории Союза по результатам
соответствующих р

C:\Users\maxik\AppData\Local\Temp\ipykernel_26244\264219195.py:89: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  results = client.search(


In [ ]:
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient

client = QdrantClient(url="http://localhost:6333")
model = SentenceTransformer("deepvk/USER-base")
collection_name = "rag_docs"

def search(query, top_k=5):
    query_embedding = model.encode([query], normalize_embeddings=True, prompt_name='query')[0]
    results = client.search(
        collection_name=collection_name,
        query_vector=query_embedding.tolist(),
        limit=top_k
    )
    return results

query = 'Если доля НС ≥30% и объем импорта из НС не падает или растет по сравнению с предыдущим периодом'
results = search(query)
display(results)


c:\Programming\PROJECTS\hackaton\trade-defense-platform\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Default prompt name is set to 'query'. This prompt will be applied to all `encode()` calls, except if `encode()` is called with `prompt` or `prompt_name` parameters.
C:\Users\maxik\AppData\Local\Temp\ipykernel_16320\1344377831.py:10: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  results = client.search(


[ScoredPoint(id='bbe4104d-a750-4ba1-b31d-4166f67adc46', version=22, score=0.51191664, payload={'filename': '5 Приложение 8 Договора о ЕАЭС.pdf', 'text': 'учае если период организации производства превышает период расследования, - за наиболее поздний \nэтап организации производства, приходящийся на период проведения расследования. \n62. Суммарные количественные показатели административных, торговых и общих издержек и прибыли, \nхарактерные для данной отрасли экономики, определяются на основе фактических данных о производстве и \nпродаже аналогичного товара при обычном ходе торговли, представляемых экспортером или \nпроизводителем товара, являющегося предметом демпингового импорта. Если такие суммарные \nколичественные показатели невозможно определить указанным образом, они могут быть определены на \nоснове: \n1) фактических сумм, полученных и израсходованных экспортером или производителем товара, \nявляющегося объектом расследования, в связи с производством и продажей той же категории т